# Week 6: Data Access (SQL) & Pandas Analysis

This notebook loads datasets from the **`week06`** schema with **local CSV caching** and **automatic offline fallback**:
- If the online Azure SQL Database is reachable, tables are queried and cached to local CSV files.
- If the online database is unreachable (e.g. IP whitelist / network changes / offline), the code automatically retrieves the local copies.

## 1. Setup & Connection

In [ ]:
import sys
import pathlib
import pandas as pd

# Adjust display options for clean DataFrame previewing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

# Ensure project root / db directory is accessible
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / 'db' / 'dataraflow_db.py').exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

from db.dataraflow_db import run_query

## 2. Table Loader with Local Caching & Offline Fallback

`get_table(table_name)` retrieves data from the local CSV if it exists; otherwise it fetches from the database and saves a local copy. If the database is offline, it automatically falls back to the local copy.

In [ ]:
def get_table(table_name: str, schema: str = 'week06', force_refresh: bool = False) -> pd.DataFrame:
    """
    Load data from local CSV if it exists; otherwise fetch from online DB and save to CSV.
    If the online DB is unreachable, automatically fall back to the local CSV copy.
    """
    csv_path = pathlib.Path(f'{table_name}.csv')

    # Use local cache if present and not forcing refresh
    if csv_path.exists() and not force_refresh:
        print(f'Loading cached data from {csv_path}...')
        return pd.read_csv(csv_path)

    # Attempt online query
    try:
        print(f'Querying online Azure SQL Database ({schema}.{table_name})...')
        df = run_query(f'SELECT * FROM {schema}.{table_name}')
        df.to_csv(csv_path, index=False)
        print(f'Successfully fetched and cached {len(df)} rows to {csv_path}')
        return df
    except Exception as e:
        print(f'Online database query failed: {e}')
        if csv_path.exists():
            print(f'Falling back to local copy at {csv_path}...')
            return pd.read_csv(csv_path)
        raise RuntimeError(f'Online database unavailable and no local copy found for {table_name}.') from e

# Backward-compatible get_data() for London 2014
def get_data(force_refresh: bool = False) -> pd.DataFrame:
    return get_table('london_2014', force_refresh=force_refresh)

## 3. Pull a Single Dataset (London 2014)

In [ ]:
london = get_table('london_2014')
print(f'London 2014 shape: {london.shape}')
london.head()

## 4. Combine the Six Cities (SQL UNION ALL with Offline pd.concat Fallback)

Combines London, Beijing, Brasília, Cape Town, Delhi, and Moscow (365 daily rows each = 2,190 rows total).
- Online: Merges using SQL `UNION ALL`.
- Offline: Merges the local CSV copies using `pd.concat`.

In [ ]:
def get_combined_cities(force_refresh: bool = False) -> pd.DataFrame:
    """
    Combine the 6 city tables using SQL UNION ALL when online,
    or fall back to combining the local CSV files using pd.concat if offline.
    """
    csv_path = pathlib.Path('cities_mean_temp.csv')

    if csv_path.exists() and not force_refresh:
        print(f'Loading cached combined cities from {csv_path}...')
        return pd.read_csv(csv_path)

    try:
        print('Querying online combined cities via SQL UNION ALL...')
        query = """
        SELECT 'London' AS city, GMT AS [Date], [Mean TemperatureC] FROM week06.london_2014
        UNION ALL
        SELECT 'Beijing', [Date], [Mean TemperatureC] FROM week06.beijing_pek_2014
        UNION ALL
        SELECT 'Brasilia', [Date], [Mean TemperatureC] FROM week06.brasilia_bsb_2014
        UNION ALL
        SELECT 'CapeTown', [Date], [Mean TemperatureC] FROM week06.capetown_cpt_2014
        UNION ALL
        SELECT 'Delhi', [Date], [Mean TemperatureC] FROM week06.delhi_del_2014
        UNION ALL
        SELECT 'Moscow', [Date], [Mean TemperatureC] FROM week06.moscow_svo_2014
        """
        cities = run_query(query)
        cities.to_csv(csv_path, index=False)
        print(f'Successfully queried and saved {len(cities)} rows to {csv_path}')
        return cities
    except Exception as e:
        print(f'Online SQL query failed: {e}')
        if csv_path.exists():
            print(f'Falling back to existing {csv_path}...')
            return pd.read_csv(csv_path)

        print('Falling back to combining individual local city CSV files with pd.concat...')
        city_configs = [
            ('London', 'london_2014', 'GMT'),
            ('Beijing', 'beijing_pek_2014', 'Date'),
            ('Brasilia', 'brasilia_bsb_2014', 'Date'),
            ('CapeTown', 'capetown_cpt_2014', 'Date'),
            ('Delhi', 'delhi_del_2014', 'Date'),
            ('Moscow', 'moscow_svo_2014', 'Date')
        ]
        dfs = []
        for city_name, table, date_col in city_configs:
            df_city = get_table(table)
            subset = pd.DataFrame({
                'city': city_name,
                'Date': df_city[date_col],
                'Mean TemperatureC': df_city['Mean TemperatureC']
            })
            dfs.append(subset)
        combined = pd.concat(dfs, ignore_index=True)
        combined.to_csv(csv_path, index=False)
        print(f'Successfully combined and saved {len(combined)} rows to {csv_path}')
        return combined

cities = get_combined_cities()
print(f'Combined cities shape: {cities.shape}')  # (2190, 3)
cities.head(10)

## 5. Aggregations (SQL AVG vs Pandas groupby)

In [ ]:
# 5.1 Average temperature in SQL
try:
    avg_by_city_sql = run_query("""
    SELECT 'London' AS city, AVG([Mean TemperatureC]) AS avg_temp FROM week06.london_2014
    UNION ALL
    SELECT 'Delhi', AVG([Mean TemperatureC]) AS avg_temp FROM week06.delhi_del_2014
    """)
    print('Average temperature (SQL):')
    print(avg_by_city_sql)
except Exception as e:
    print(f'SQL aggregation skipped ({e}); using Pandas aggregation.')

# 5.2 Average temperature in Pandas (works offline too)
avg_by_city_pandas = cities.groupby('city')['Mean TemperatureC'].mean().reset_index()
print('\nAverage temperature (Pandas groupby):')
print(avg_by_city_pandas)

## 6. WHO Population & TB Datasets

In [ ]:
who_all = get_table('who_pop_tb_all')
print(f'WHO All shape: {who_all.shape}')
who_all.head()

In [ ]:
who_some = get_table('who_pop_tb_some')
print(f'WHO Some shape: {who_some.shape}')
who_some.head()